# LLM fine tuning for Qwen2.5-3B-Instruct

In [1]:
! pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [2]:
! pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [3]:
"""
train_lora.py

LoRA fine-tuning of Qwen2.5-3B-Instruct on your generated Q&A dataset.
Designed to run in Google Colab (free tier, T4 GPU) as a single script
pasted into cells, or run top-to-bottom in one cell.

Uses full fp16/bf16 LoRA (no quantization) — fits on a T4 for a 3B model.

Expects train.jsonl and val.jsonl already uploaded to your Colab session
or Google Drive, in the chat-message format:
    {"messages": [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", ...}]}
"""

# ============================================================
# CELL 1 — Install dependencies
# ============================================================
# Run this first in Colab (uncomment if running as a script directly):
#
# !pip install -q -U transformers accelerate peft trl datasets bitsandbytes
#
# Colab sometimes ships a torchao version that's incompatible with peft's
# import checks, causing:
#   ImportError: Found an incompatible version of torchao...
# Fix: remove it (LoRA/QLoRA here don't need it) — uncomment the line below.
#
# !pip uninstall -y torchao
#
# IMPORTANT: after running this cell, go to Runtime -> Restart runtime,
# then run all cells again from the top before proceeding.

# ============================================================
# CELL 2 — Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os

# Where your train.jsonl / val.jsonl live. Adjust to wherever you put them
# (e.g. if you uploaded them straight to Drive under a project folder).
DATA_DIR = "/content/drive/MyDrive/llm-finetuning/dataset"
TRAIN_FILE = os.path.join(DATA_DIR, "train.jsonl")
VAL_FILE = os.path.join(DATA_DIR, "val.jsonl")

# Where the final LoRA adapter + tokenizer get saved on Drive
OUTPUT_DIR_DRIVE = "/content/drive/MyDrive/llm-finetuning/qwen2.5-3b-lora"

# Local scratch dir for checkpoints during training (faster than writing to Drive every step)
LOCAL_OUTPUT_DIR = "/content/qwen2.5-3b-lora-checkpoints"

os.makedirs(OUTPUT_DIR_DRIVE, exist_ok=True)
os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)

# ============================================================
# CELL 3 — Imports
# ============================================================
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

# ------------------------------------------------------------
# GPU CHECK — do this before anything else. If this prints False,
# STOP and go to Runtime -> Change runtime type -> T4 GPU, then
# re-run from the top. Training a 3B model on CPU is impractically
# slow (hours per step instead of seconds).
# ------------------------------------------------------------
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU detected! Go to Runtime -> Change runtime type -> "
        "Hardware accelerator -> T4 GPU, then re-run this notebook."
    )

# ============================================================
# CELL 4 — Config
# ============================================================
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

MAX_SEQ_LENGTH = 1024
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUMULATION_STEPS = 8       # effective batch size = 2 * 8 = 16
SAVE_STEPS = 50                   # checkpoint frequency (protects against Colab disconnects)
LOGGING_STEPS = 10
EVAL_STEPS = 50

# ============================================================
# CELL 5 — Load dataset
# ============================================================
dataset = load_dataset(
    "json",
    data_files={"train": TRAIN_FILE, "validation": VAL_FILE},
)
print(dataset)
print("\nExample record:")
print(dataset["train"][0])

# ============================================================
# CELL 6 — Load tokenizer and model (fp16/bf16, no quantization)
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,   # T4 supports bf16 compute fine for this size
    device_map="auto",
)
model.config.use_cache = False  # required for gradient checkpointing during training

# ============================================================
# CELL 7 — LoRA config
# ============================================================
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ============================================================
# CELL 8 — Format function (applies the model's chat template)
# ============================================================
def formatting_func(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return text

# ============================================================
# CELL 9 — Training arguments
# ============================================================
# NOTE: max sequence length is set via tokenizer.model_max_length below
# instead of an SFTConfig kwarg, since trl has renamed that argument
# across versions (max_seq_length vs max_length). This avoids the
# version mismatch entirely.
tokenizer.model_max_length = MAX_SEQ_LENGTH

sft_config = SFTConfig(
    output_dir=LOCAL_OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    bf16=True,
    packing=False,
    report_to="none",
)

# ============================================================
# CELL 10 — Trainer
# ============================================================
import inspect

trainer_params = set(inspect.signature(SFTTrainer.__init__).parameters.keys())
tokenizer_kwarg = "processing_class" if "processing_class" in trainer_params else "tokenizer"

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    formatting_func=formatting_func,
    **{tokenizer_kwarg: tokenizer},
)

# ============================================================
# CELL 11 — Train
# ============================================================
trainer.train()

# ============================================================
# CELL 12 — Save final model to Google Drive
# ============================================================
# Save LoRA adapter + tokenizer (small — a few hundred MB, not the full base model)
model.save_pretrained(OUTPUT_DIR_DRIVE)
tokenizer.save_pretrained(OUTPUT_DIR_DRIVE)

print(f"\nLoRA adapter saved to: {OUTPUT_DIR_DRIVE}")
print("Files:", os.listdir(OUTPUT_DIR_DRIVE))

# ============================================================
# OPTIONAL — also save the merged full model (base + adapter combined)
# Uncomment if you want a standalone model instead of an adapter.
# Note: this is much larger (~6GB) and takes extra time/memory to merge.
# ============================================================
# from peft import PeftModel
# merged_model = model.merge_and_unload()
# MERGED_OUTPUT_DIR = "/content/drive/MyDrive/llm-finetuning/qwen2.5-3b-lora-merged"
# os.makedirs(MERGED_OUTPUT_DIR, exist_ok=True)
# merged_model.save_pretrained(MERGED_OUTPUT_DIR)
# tokenizer.save_pretrained(MERGED_OUTPUT_DIR)
# print(f"Merged model saved to: {MERGED_OUTPUT_DIR}")

Mounted at /content/drive
CUDA available: True
GPU: Tesla T4


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 495
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 55
    })
})

Example record:
{'messages': [{'role': 'system', 'content': 'You are a helpful home-health assistant. You give general, safe, non-diagnostic guidance for common everyday illnesses. You always recommend seeing a doctor for severe, worsening, or unusual symptoms. You do not provide specific medication dosages or diagnose conditions.'}, {'role': 'user', 'content': 'What are some common causes of stomach ache and indigestion in adults, and how can I avoid them?'}, {'role': 'assistant', 'content': 'Common causes include eating too quickly, not chewing food properly, or consuming foods high in acidity, spice, or fat. Try eating slowly, avoiding trigger foods, and staying hydrated. If your symptoms worsen or last for an extended period, see a doctor.'}]}


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

Building labels for eval dataset:   0%|          | 0/55 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/55 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.531983,0.537699,0.531358,128335.000000,0.837191
93,0.447777,0.527789,0.470073,237942.000000,0.837504



LoRA adapter saved to: /content/drive/MyDrive/llm-finetuning/qwen2.5-3b-lora
Files: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']


# LLM fine tuning for DeepSeek-R1-Distill-Qwen-7B

In [1]:
!pip install -U -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.1 MB/s eta 0:00:00


In [2]:
! pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 15.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [3]:
import bitsandbytes as bnb
print(bnb.__version__)

0.49.2


In [4]:
"""
train_qlora.py

QLoRA fine-tuning of DeepSeek-R1-Distill-Qwen-7B on your generated Q&A dataset.
Designed to run in Google Colab (free tier, T4 GPU) as a single script
pasted into cells, or run top-to-bottom in one cell.

Uses 4-bit quantization (QLoRA) — this is what makes a 7B model fit on a
free T4 (16GB). Mirrors train_lora.py's structure so the two are easy to
compare side by side.

Expects train.jsonl and val.jsonl already uploaded to Google Drive, in the
chat-message format:
    {"messages": [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", ...}]}
"""

# ============================================================
# CELL 1 — Install dependencies
# ============================================================
# Run this first in Colab:
#
# !pip install -q -U transformers accelerate peft trl datasets bitsandbytes
#
# Colab sometimes ships a torchao version incompatible with peft's import
# checks (ImportError: Found an incompatible version of torchao...).
# LoRA/QLoRA here don't need torchao, so remove it if you hit that error:
#
# !pip uninstall -y torchao
#
# IMPORTANT: after running this cell (especially the uninstall line), go to
# Runtime -> Restart runtime, then run all cells again from the top.

# ============================================================
# CELL 2 — Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os

# Where your train.jsonl / val.jsonl live.
DATA_DIR = "/content/drive/MyDrive/llm-finetuning/dataset"
TRAIN_FILE = os.path.join(DATA_DIR, "train.jsonl")
VAL_FILE = os.path.join(DATA_DIR, "val.jsonl")

# Where the final QLoRA adapter + tokenizer get saved on Drive
OUTPUT_DIR_DRIVE = "/content/drive/MyDrive/llm-finetuning/deepseek-r1-7b-qlora"

# Local scratch dir for checkpoints during training
LOCAL_OUTPUT_DIR = "/content/deepseek-r1-7b-qlora-checkpoints"

os.makedirs(OUTPUT_DIR_DRIVE, exist_ok=True)
os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)

# ============================================================
# CELL 3 — Imports + GPU check
# ============================================================
import torch
import inspect
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

# GPU CHECK — do this before anything else. A 7B model, even 4-bit, is not
# practical on CPU. If this raises, go to Runtime -> Change runtime type ->
# Hardware accelerator -> T4 GPU, then re-run from the top.
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU detected! Go to Runtime -> Change runtime type -> "
        "Hardware accelerator -> T4 GPU, then re-run this notebook."
    )

# ============================================================
# CELL 4 — Config
# ============================================================
BASE_MODEL = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

MAX_SEQ_LENGTH = 1024
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH_SIZE = 1          # smaller than the LoRA script — 7B in 4-bit still needs headroom
GRAD_ACCUMULATION_STEPS = 16       # effective batch size = 1 * 16 = 16
SAVE_STEPS = 50
LOGGING_STEPS = 10
EVAL_STEPS = 50
EARLY_STOPPING_PATIENCE = 3        # stop if eval loss doesn't improve for 3 eval rounds in a row

# ============================================================
# CELL 5 — Load dataset
# ============================================================
dataset = load_dataset(
    "json",
    data_files={"train": TRAIN_FILE, "validation": VAL_FILE},
)
print(dataset)
print("\nExample record:")
print(dataset["train"][0])

# ============================================================
# CELL 6 — Load tokenizer and model in 4-bit (QLoRA)
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False

# Required prep step for k-bit (4-bit) training — enables gradient checkpointing
# and casts certain layers appropriately for stable QLoRA training.
model = prepare_model_for_kbit_training(model)

# ============================================================
# CELL 7 — LoRA config (same target modules/rank as the LoRA script,
# so the two runs are a fair comparison)
# ============================================================
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ============================================================
# CELL 8 — Format function (applies the model's chat template)
# ============================================================
def formatting_func(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return text

# ============================================================
# CELL 9 — Training arguments
# ============================================================
# max sequence length is set via tokenizer.model_max_length rather than an
# SFTConfig kwarg, since trl has renamed that argument across versions.
tokenizer.model_max_length = MAX_SEQ_LENGTH

sft_config = SFTConfig(
    output_dir=LOCAL_OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    bf16=True,
    packing=False,
    report_to="none",
    load_best_model_at_end=True,       # needed for early stopping to restore the best checkpoint
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# ============================================================
# CELL 10 — Trainer (with early stopping)
# ============================================================
# trl renamed the tokenizer argument to processing_class in newer versions.
# Detect which one this installed version expects.
trainer_params = set(inspect.signature(SFTTrainer.__init__).parameters.keys())
tokenizer_kwarg = "processing_class" if "processing_class" in trainer_params else "tokenizer"

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    formatting_func=formatting_func,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    **{tokenizer_kwarg: tokenizer},
)

# ============================================================
# CELL 11 — Train
# ============================================================
trainer.train()

# ============================================================
# CELL 12 — Save final model to Google Drive
# ============================================================
# Saves the LoRA adapter only (small, a few hundred MB) — NOT the merged
# full model. This is the recommended default: the frozen 4-bit base model
# stays as-is, and only the small trained adapter needs to be saved/shared.
model.save_pretrained(OUTPUT_DIR_DRIVE)
tokenizer.save_pretrained(OUTPUT_DIR_DRIVE)

print(f"\nQLoRA adapter saved to: {OUTPUT_DIR_DRIVE}")
print("Files:", os.listdir(OUTPUT_DIR_DRIVE))

# ============================================================
# NOTE on merging for QLoRA
# ============================================================
# Merging a QLoRA adapter back into a full-precision model is possible but
# more involved than the plain-LoRA case (you can't simply merge into a
# 4-bit model — you'd need to reload the base model in fp16/bf16 first,
# then merge). Given free-tier Colab's limited RAM/VRAM, it's best to just
# keep the adapter and load it on top of the base model at inference time
# with peft's PeftModel.from_pretrained(), rather than merging.

Mounted at /content/drive
CUDA available: True
GPU: Tesla T4


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 495
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 55
    })
})

Example record:
{'messages': [{'role': 'system', 'content': 'You are a helpful home-health assistant. You give general, safe, non-diagnostic guidance for common everyday illnesses. You always recommend seeing a doctor for severe, worsening, or unusual symptoms. You do not provide specific medication dosages or diagnose conditions.'}, {'role': 'user', 'content': 'What are some common causes of stomach ache and indigestion in adults, and how can I avoid them?'}, {'role': 'assistant', 'content': 'Common causes include eating too quickly, not chewing food properly, or consuming foods high in acidity, spice, or fat. Try eating slowly, avoiding trigger foods, and staying hydrated. If your symptoms worsen or last for an extended period, see a doctor.'}]}


config.json:   0%|          | 0.00/680 [00:00<?, ?B/s]

...

Building labels for eval dataset:   0%|          | 0/55 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/55 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 151646, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.805198,0.814225,0.814092,119546.000000,0.788214
93,0.678597,0.787651,0.716240,221607.000000,0.796783



QLoRA adapter saved to: /content/drive/MyDrive/llm-finetuning/deepseek-r1-7b-qlora
Files: ['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']


# compare before after

In [1]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


# LoRA before after

In [3]:
"""
compare_before_after.py

Loads a base model and your fine-tuned adapter (LoRA or QLoRA) on top of it,
then runs the same set of test questions through both so you can see exactly
what the fine-tuning changed. This is your main showcase artifact.

Run this in Colab, in its own fresh session (don't mix with a training run —
loading two copies of the model at once needs more VRAM than training alone).

Works for EITHER of your two fine-tunes — just set ADAPTER_TYPE below.
"""

# ============================================================
# CELL 1 — Install dependencies
# ============================================================
# !pip install -q -U transformers accelerate peft bitsandbytes
# !pip uninstall -y torchao   # uncomment if you hit the torchao ImportError again

# ============================================================
# CELL 2 — Mount Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# CELL 3 — Config — pick which fine-tune you're comparing
# ============================================================
# Set this to "lora" or "qlora" depending on which adapter you want to test.
ADAPTER_TYPE = "lora"   # change to "qlora" to test the other one

if ADAPTER_TYPE == "lora":
    BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
    ADAPTER_PATH = "/content/drive/MyDrive/llm-finetuning/qwen2.5-3b-lora"
    USE_4BIT = False
elif ADAPTER_TYPE == "qlora":
    BASE_MODEL = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
    ADAPTER_PATH = "/content/drive/MyDrive/llm-finetuning/deepseek-r1-7b-qlora"
    USE_4BIT = True
else:
    raise ValueError("ADAPTER_TYPE must be 'lora' or 'qlora'")

SYSTEM_PROMPT = (
    "You are a helpful home-health assistant. You give general, safe, "
    "non-diagnostic guidance for common everyday illnesses. You always "
    "recommend seeing a doctor for severe, worsening, or unusual symptoms. "
    "You do not provide specific medication dosages or diagnose conditions."
)

# Test questions — mix of ones similar to your training topics AND a couple
# slightly different, to show the model generalizes rather than just
# memorizing exact training examples.
TEST_QUESTIONS = [
    "What should I do if I have a mild fever?",
    "My child has a wet cough, what can help?",
    "I have a pounding headache, any home remedies?",
    "How do I deal with a sore throat at home?",
    "What can I do about mild constipation?",
    "Is it normal to feel very tired with a cold?",  # slightly novel phrasing
]

MAX_NEW_TOKENS = 200

# ============================================================
# CELL 4 — Imports + GPU check
# ============================================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU detected! Set Runtime -> T4 GPU before running.")

# ============================================================
# CELL 5 — Load tokenizer
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ============================================================
# CELL 6 — Load base model (used for the "before" answers)
# ============================================================
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb_config, device_map="auto"
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto"
    )

# ============================================================
# CELL 7 — Load the fine-tuned version (base + your adapter, for "after")
# ============================================================
finetuned_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

# ============================================================
# CELL 8 — Generation helper
# ============================================================
def generate_answer(model, question: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# ============================================================
# CELL 9 — Run comparison
# ============================================================
# IMPORTANT: to get a genuine "before" answer, we temporarily disable the
# adapter on the SAME model object (base_model and finetuned_model share
# the same underlying weights — finetuned_model just adds the adapter on
# top). Using .disable_adapter() avoids needing two separate model copies
# in memory, which matters on a free-tier T4.

results = []

for question in TEST_QUESTIONS:
    print(f"\n{'='*70}\nQ: {question}\n{'='*70}")

    with finetuned_model.disable_adapter():
        before_answer = generate_answer(finetuned_model, question)
    print(f"\n[BEFORE - base model]\n{before_answer}")

    after_answer = generate_answer(finetuned_model, question)
    print(f"\n[AFTER - fine-tuned]\n{after_answer}")

    results.append({
        "question": question,
        "before": before_answer,
        "after": after_answer,
    })

# ============================================================
# CELL 10 — Save results to Drive as a readable file
# ============================================================
import json

RESULTS_PATH = f"/content/drive/MyDrive/llm-finetuning/comparison_{ADAPTER_TYPE}.json"
with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\n\nSaved comparison results to: {RESULTS_PATH}")

# Also print a clean markdown-style summary you can paste straight into a
# README or portfolio writeup.
print("\n\n" + "="*70)
print("MARKDOWN SUMMARY (copy this into your README)")
print("="*70)
for r in results:
    print(f"\n**Q: {r['question']}**\n")
    print(f"- Before: {r['before']}\n")
    print(f"- After: {r['after']}\n")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CUDA available: True
GPU: Tesla T4


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


Q: What should I do if I have a mild fever?

[BEFORE - base model]
If you have a mild fever, it's important to monitor your overall health and take some basic steps to feel better. Here’s what you can consider doing:

1. **Stay Hydrated**: Drink plenty of fluids like water, herbal teas (avoid those with caffeine), or clear broths to help prevent dehydration.

2. **Rest**: Give your body the chance to recover by getting adequate rest. Sleep is very beneficial when you're feeling unwell.

3. **Comfort Measures**: Use a cool compress on your forehead or take a lukewarm bath to help reduce your fever. Dress in light clothing.

4. **Over-the-Counter Medications**: You can use over-the-counter medications such as acetaminophen (Tylenol) or ibuprofen (Advil) to help manage the fever and any discomfort. Make sure to follow the dosage instructions carefully.

5. **Hydration and Nutrition**: Eat nutritious foods if you can. Sometimes bland, easy-to-digest foods

[AFTER - fine-tuned]
For a mild 

# QLoRA before after

In [1]:
!pip install -U bitsandbytes>=0.46.1

In [2]:
"""
compare_before_after.py

Loads a base model and your fine-tuned adapter (LoRA or QLoRA) on top of it,
then runs the same set of test questions through both so you can see exactly
what the fine-tuning changed. This is your main showcase artifact.

Run this in Colab, in its own fresh session (don't mix with a training run —
loading two copies of the model at once needs more VRAM than training alone).

Works for EITHER of your two fine-tunes — just set ADAPTER_TYPE below.
"""

# ============================================================
# CELL 1 — Install dependencies
# ============================================================
# !pip install -q -U transformers accelerate peft bitsandbytes
# !pip uninstall -y torchao   # uncomment if you hit the torchao ImportError again

# ============================================================
# CELL 2 — Mount Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# CELL 3 — Config — pick which fine-tune you're comparing
# ============================================================
# Set this to "lora" or "qlora" depending on which adapter you want to test.
ADAPTER_TYPE = "qlora"   # change to "qlora" to test the other one

if ADAPTER_TYPE == "lora":
    BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
    ADAPTER_PATH = "/content/drive/MyDrive/llm-finetuning/qwen2.5-3b-lora"
    USE_4BIT = False
elif ADAPTER_TYPE == "qlora":
    BASE_MODEL = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
    ADAPTER_PATH = "/content/drive/MyDrive/llm-finetuning/deepseek-r1-7b-qlora"
    USE_4BIT = True
else:
    raise ValueError("ADAPTER_TYPE must be 'lora' or 'qlora'")

SYSTEM_PROMPT = (
    "You are a helpful home-health assistant. You give general, safe, "
    "non-diagnostic guidance for common everyday illnesses. You always "
    "recommend seeing a doctor for severe, worsening, or unusual symptoms. "
    "You do not provide specific medication dosages or diagnose conditions."
)

# Test questions — mix of ones similar to your training topics AND a couple
# slightly different, to show the model generalizes rather than just
# memorizing exact training examples.
TEST_QUESTIONS = [
    "What should I do if I have a mild fever?",
    "My child has a wet cough, what can help?",
    "I have a pounding headache, any home remedies?",
    "How do I deal with a sore throat at home?",
    "What can I do about mild constipation?",
    "Is it normal to feel very tired with a cold?",  # slightly novel phrasing
]

MAX_NEW_TOKENS = 200

# ============================================================
# CELL 4 — Imports + GPU check
# ============================================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU detected! Set Runtime -> T4 GPU before running.")

# ============================================================
# CELL 5 — Load tokenizer
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ============================================================
# CELL 6 — Load base model (used for the "before" answers)
# ============================================================
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb_config, device_map="auto"
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto"
    )

# ============================================================
# CELL 7 — Load the fine-tuned version (base + your adapter, for "after")
# ============================================================
finetuned_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

# ============================================================
# CELL 8 — Generation helper
# ============================================================
def generate_answer(model, question: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# ============================================================
# CELL 9 — Run comparison
# ============================================================
# IMPORTANT: to get a genuine "before" answer, we temporarily disable the
# adapter on the SAME model object (base_model and finetuned_model share
# the same underlying weights — finetuned_model just adds the adapter on
# top). Using .disable_adapter() avoids needing two separate model copies
# in memory, which matters on a free-tier T4.

results = []

for question in TEST_QUESTIONS:
    print(f"\n{'='*70}\nQ: {question}\n{'='*70}")

    with finetuned_model.disable_adapter():
        before_answer = generate_answer(finetuned_model, question)
    print(f"\n[BEFORE - base model]\n{before_answer}")

    after_answer = generate_answer(finetuned_model, question)
    print(f"\n[AFTER - fine-tuned]\n{after_answer}")

    results.append({
        "question": question,
        "before": before_answer,
        "after": after_answer,
    })

# ============================================================
# CELL 10 — Save results to Drive as a readable file
# ============================================================
import json

RESULTS_PATH = f"/content/drive/MyDrive/llm-finetuning/comparison_{ADAPTER_TYPE}.json"
with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\n\nSaved comparison results to: {RESULTS_PATH}")

# Also print a clean markdown-style summary you can paste straight into a
# README or portfolio writeup.
print("\n\n" + "="*70)
print("MARKDOWN SUMMARY (copy this into your README)")
print("="*70)
for r in results:
    print(f"\n**Q: {r['question']}**\n")
    print(f"- Before: {r['before']}\n")
    print(f"- After: {r['after']}\n")

Mounted at /content/drive
CUDA available: True
GPU: Tesla T4


config.json:   0%|          | 0.00/680 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]


Q: What should I do if I have a mild fever?


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



[BEFORE - base model]
Okay, so I have a mild fever, and I'm not sure what to do. I remember hearing that fevers can be caused by many things, but I'm not exactly a medical expert. Let me think through this step by step.

First, what causes a mild fever? I think it could be a virus, maybe the flu, or perhaps something else. I've heard that a slight fever isn't necessarily a sign of a serious illness. But I'm not entirely sure. Maybe it's just a common cold or something like that.

I should consider the symptoms alongside the fever. Do I have any other symptoms like a cough, runny nose, sore throat, or maybe fatigue? If I do, that might help determine what's going on. But since I'm only experiencing a mild fever, maybe it's just a minor infection.

What should I do about the fever itself? I don't want to take unnecessary medications. I know that acetaminophen or ibuprofen

[AFTER - fine-tuned]
For a mild fever, try resting, drinking plenty of fluids, and using a fever reducer or antihis